In [1]:
# Install VibeVoice
!pip install -q vibevoice soundfile

In [2]:
# Patch VibeVoice: add None-guards for speech_tensors/speech_masks in the prefill block.
# Uses find_spec (not import) to locate the file so the module isn't cached before patching.
import importlib.util, pathlib

_spec = importlib.util.find_spec("vibevoice.modular.modeling_vibevoice_inference")
_path = pathlib.Path(_spec.origin)
_src = _path.read_text()

_patches = {
    '"speech_tensors": speech_tensors.to(device=device),':
        '"speech_tensors": speech_tensors.to(device=device) if speech_tensors is not None else None,',
    '"speech_masks": speech_masks.to(device),':
        '"speech_masks": speech_masks.to(device) if speech_masks is not None else None,',
    '"speech_input_mask": speech_input_mask.to(device),':
        '"speech_input_mask": speech_input_mask.to(device) if speech_input_mask is not None else None,',
}

for old, new in _patches.items():
    if old in _src:
        _src = _src.replace(old, new)
        print(f"Patched: {old.strip()}")
    elif new in _src:
        print(f"Already patched: {old.strip()}")
    else:
        print(f"NOT FOUND — version mismatch: {old.strip()}")

_path.write_text(_src)
print("File written. Run the cells below fresh (no restart needed).")

In [3]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — switch Runtime to T4")

In [4]:
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

MODEL_ID = "vibevoice/VibeVoice-1.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_ID} on {device}...")
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).to(device)
model.eval()
print("Model ready.")

In [6]:
script = """\
speaker 1: Kate says she's saving herself—classic \n
speaker 2: Well actually, Lost treats her "escape mindset" like the Island has a billing department. \n
speaker 1: You think you're leaving and the Island's like, Cute. Anyway. \n
speaker 2: Hey, Maya. Welcome back to Lost Deep Dive. Today we're doing a Kate-focused dive into how Lost uses her motives—escape, control, protection, guilt—to build mythology. \n
speaker 1: And I'm already bracing for the part where we're told she's just reacting. \n
"""


In [7]:
inputs = processor(text=script, return_tensors="pt", padding=True)
inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

print("Generating audio...")
with torch.inference_mode():
    output = model.generate(
        **inputs,
        tokenizer=processor.tokenizer,
        return_speech=True,
    )
print("Done.")

In [8]:
import soundfile as sf
from IPython.display import Audio, display

SAMPLE_RATE = 24000
OUT_FILE = "episode_clip.wav"

chunks = [c for c in output.speech_outputs if c is not None]
audio = torch.cat(chunks, dim=-1).squeeze().cpu().float().numpy()

sf.write(OUT_FILE, audio, SAMPLE_RATE)
print(f"Saved {len(audio)/SAMPLE_RATE:.1f}s → {OUT_FILE}")
display(Audio(audio, rate=SAMPLE_RATE))